In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

#download and read the text file from the URL
with urllib.request.urlopen(url) as response:
    text = response.read().decode('utf-8')

#verify
print(f"Dataset length: {len(text)} characters")
print("\nFirst 100 characters:")
print(text[:100])


Dataset length: 1115394 characters

First 100 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [2]:
#all the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [3]:
#creating decoders and encoders for the characters
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # this is the encoder, it takes a string and outputs a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # this is the decoder, it takes a list of integers and outputs a string
#stoi= s to i, string to integer and itos is the opposite

print(encode("hey, what are you doing?"))
print(decode(encode("hey, what are you doing?")))


[46, 43, 63, 6, 1, 61, 46, 39, 58, 1, 39, 56, 43, 1, 63, 53, 59, 1, 42, 53, 47, 52, 45, 12]
hey, what are you doing?


In [4]:
# encoding the entire text dataset and storing it in a tensor
import torch 

data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:100])


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [5]:
#splitting up the data into train and validation sets
n = int(0.9*len(data)) #first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [6]:
#we're going to train in chunks
block_size = 8 #how many characters to consider at once, or one chunk
print(train_data[:block_size+1]) #we add 1 because we want to see the next character as well, so we can predict it


x = train_data[:block_size] #first 8 characters
y = train_data[1:block_size+1] #next 8 characters, so we can predict the next character
for t in range(block_size):
    context = x[:t+1] #the context is the first t+1 characters
    target = y[t] #the target is the next character
    print(f"when input is {context} the target: {target}")

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [7]:
torch.manual_seed(1337) #for reproducibility
batch_size = 4 #how many independent sequences will we process in parallel
block_size = 8 #what is the maximum context length for predictions

def get_batch(split): #generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data #if split == train, then data = train_data, else data = val_data (sleek)
    ix = torch.randint(len(data) - block_size, (batch_size,)) #randomly select batch_size starting indices for the sequences
    x = torch.stack([data[i:i+block_size] for i in ix]) # for each starting index, get the next block_size characters and stack them into a tensor
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # same as x but with an offset of 1, so we can predict the next character
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("\ntargets:")
print(yb.shape)
print(yb)

print('------')

for b in range(batch_size): #batch dimension
    for t in range(block_size): #time dimension
        context = xb[b, :t+1] #the context is the first t+1 characters of the b-th sequence in the batch
        target = yb[b, t] #the target is the next character in the b-th sequence
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44,

In [8]:
xb # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337) #for reproducibility

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch,Time,Channel) where Channel is the vocab size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # To make it compatible with the loss function, we reshape it to (Batch*Time, Channel) so it better fits the pytorch input for the loss function, which expects a 2D tensor of shape (N,C) where N is the number of samples and C is the number of classes
            targets = targets.view(B*T) # same as above, we reshape the targets to (Batch*Time,) so it better fits the pytorch input for the loss function, which expects a 1D tensor of shape (N,) where N is the number of samples

            loss = F.cross_entropy(logits, targets) # negative log likelihood loss, we want to minimize this loss, so we want to maximize the probability of the correct next character
            # above it measures the quality of the logits in respective to the targets
            
        return logits, loss   # basically the scores for the next chatracter in the sequence

    def generate(self, idx, max_new_tokens): #job of generate is to take the (B,T) and extend it to (B,T+max_new_tokens) by predicting the next token max_new_tokens times
        # idex is (b,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self(idx)
            #focus only on the last time step
            logits = logits[:, -1, :] #becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape) # (batch_size, block_size, vocab_size)
print(loss) # a single number, the loss for the batch
# ideally the loss should be around log(vocab_size) = -ln(1/vocab_size) = -ln(1/65) = 4.17, so there is some entropy in the model, but not too much
idx = torch.zeros((1,1), dtype=torch.long) # Batch will be 1, and Time will be 1, so we start with a single token and data type will be long because we are dealing with indices
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) # decode the generated indices to get the generated text


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [10]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) # AdamW is a variant of Adam that decouples weight decay from the gradient update, which is better for training transformers

In [11]:
batch_size = 32
for steps in range(10000): 

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # set_to_none=True is a more efficient way to zero the gradients, it avoids creating a new tensor and instead sets the existing tensor to None
    loss.backward() # backpropagate the loss
    optimizer.step() # update the parameters

print(loss.item()) # print the loss for this batch


#So in here, we increase batch size to 32, and we run for 100 steps.
#we sample a batch of data, evaluate the loss, zeroing out all the gradients from the previous step getting the gradients for all the parameters and then update our parameters



2.382369041442871


In [12]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=600)[0].tolist()))
# we see a dramatic improvement from previous runs and increasing the tokens also help
# This rn is a very simple model, because the tokens are not talking to eachother


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th likl's amen, tofr,
n s Byo tred ceathe, il ivilde w
O ff y
Fivede? ig aiMy, I ivis muofounce herevern 


### The mathematical trick in self-attention

In [13]:
# consider the following example

torch.manual_seed(1337)
B,T,C = 4,8,2 #batch, time and channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

We need to make sure that the tokens communicate with eachother but it should be flowing only one way, i.e., the 5th token should not communicate with 
6th or 7th token, it should only do that with the previous ones.

In [14]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] #(t,C)
        xbow[b,t] = torch.mean(xprev, 0)
# we're just averaging out everything previous

In [15]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [16]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

Looking above, you can see the difference between X and Xbow as each one in xbow is an average of the ones before.
But this is extremely inefficient. So we don't do like above

In [17]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a@b
print('a=')
print(a)
print('———')
print('b=')
print(b)
print('———')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
———
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
———
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


AS seen above, we can do the averages in the incremental fashion

In [18]:
#ver 2
weights = torch.tril(torch.ones(T,T))
weights = weights/weights.sum(1, keepdim=True) #this will be our "a" in the abive example
xbow2 = weights @ x #(B,T,T) @ (B,T,C) -> (B,T,C)
torch.allclose(xbow, xbow2, atol=1e-5)

True

In [19]:
xbow[0], xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [20]:
#ver 3: use softmax
tril = torch.tril(torch.ones(T,T))
weights = torch.zeros((T,T)) #weights will all be 0
weights = weights.masked_fill(tril==0, float('-inf')) # wherever the tril is 0, make it negative infinity
weights = F.softmax(weights, dim=-1) #do softmax on every single row
xbow3 = weights @ x
torch.allclose(xbow, xbow3, atol=1e-5)

True

In [21]:
#ver 4: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 #batch, time, chanels
x = torch.randn(B,T,C)

#let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) #(B, T, 16)
q = query(x) #(B, T, 16)

weights = q@k.transpose(-2,-1) #(B,T,16) @ (B, 16, T) --> (B,T,T)

tril = torch.tril(torch.ones(T,T))

weights = weights.masked_fill(tril==0, float('-inf')) 
weights = F.softmax(weights, dim=-1) 

v = value(x)
out = weights @ v


out.shape

torch.Size([4, 8, 16])

Every single token at each posiion will emit 2 vectors, a query and a key. Query is basially "what am I looking for" while key is like "what do I contain" and the way we get the affinities between these tokens is basically we just do a dot ptoduct between the keys and the queries. So my query dot products with all the keys of all the other tokens and now that is stored as weights. If the key and query are self aligned, they'll interact with very high amnt and then we will get to know about thats specific token compared to any other token in the sequence.

Pointers:
1. Attention is a communication mechanism. It can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependednt weights.
2. There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens
3. Each example across batch dimension is processed completely independently and never "talk" to each other
4. Above is "self-attention" because the keys, queries anf the values are all coming from the same source,x.

In [22]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
weights = q @ k.transpose(-2,-1) * head_size**-0.5 #scaling is used to control the varience

In [23]:
k.var()

tensor(1.0449)

In [24]:
q.var()

tensor(1.0700)

In [25]:
weights.var()

tensor(1.0918)

In [26]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim =-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [27]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim =-1) #we can see due ti the softmax sharpen towards the max

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [34]:
#copying this code from his make more series
class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    
  
  def __call__(self, x):
    # calculate the forward pass
    
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32,100) #batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [35]:
x[:,0].mean(), x[:,0].std() #mean, std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [36]:
x[0,:].mean(), x[0,:].std() #mean, std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))